# View Production Output

Load and visualize events from production batch output files (resp/seg/corr).
No simulation needed — everything is read from the HDF5 files.

**Required:** Only the output directory from `run_batch.py`.

## Setup

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

PRODUCTION_DIR = '../production_run'  # directory containing resp/, seg/, corr/
DATASET = 'production'               # dataset name used in run_batch.py
FILE_INDEX = 0                        # which file (0000, 0001, ...)
EVENT_INDEX = 7                       # event within the file (0-indexed)

# Visualization
THRESHOLD_ENC = 500                   # threshold in electrons for signal display

print(f'Production dir: {PRODUCTION_DIR}')
print(f'Dataset: {DATASET}, file: {FILE_INDEX:04d}, event: {EVENT_INDEX}')

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

import numpy as np
import matplotlib.pyplot as plt

from production.load import (
    get_file_paths, load_config, build_viz_config,
    load_event_resp, load_event_seg, load_event_corr,
    PLANE_KEYS, PLANE_NAMES,
)
from tools.visualization import (
    visualize_wire_signals,
    visualize_diffused_charge,
    visualize_track_labels,
    get_top_tracks_by_charge,
)

In [ ]:
# =============================================================================
# FILE PATHS AND METADATA
# =============================================================================

resp_path, seg_path, corr_path = get_file_paths(PRODUCTION_DIR, DATASET, FILE_INDEX)

has_resp = os.path.exists(resp_path)
has_seg = os.path.exists(seg_path)
has_corr = os.path.exists(corr_path)

print(f'resp: {"found" if has_resp else "MISSING"}  ({resp_path})')
print(f'seg:  {"found" if has_seg else "MISSING"}')
print(f'corr: {"found" if has_corr else "MISSING"}')

# Load metadata and build viz config
meta = load_config(resp_path)
viz_config = build_viz_config(resp_path)

num_time_steps = meta['num_time_steps']
threshold_adc_file = meta['threshold_adc']
electrons_per_adc = meta['electrons_per_adc']

print(f'\nEvents in file: {meta["n_events"]}')
print(f'Grid: {num_time_steps} time x {meta["time_step_us"]} us/bin')
print(f'File threshold: {threshold_adc_file:.2f} ADC ({threshold_adc_file * electrons_per_adc:.0f} e-)')
print(f'Physics: v={meta["velocity_cm_us"]} cm/us, tau={meta["lifetime_us"]} us, recomb={meta["recombination_model"]}')
print(f'Pipeline: noise={meta["include_noise"]}, electronics={meta["include_electronics"]}, digitize={meta["include_digitize"]}')

## Load and Visualize Response Signals

In [ ]:
# =============================================================================
# LOAD RESPONSE SIGNALS
# =============================================================================

dense_signals, event_attrs = load_event_resp(resp_path, EVENT_INDEX)

n_deposits = int(event_attrs['n_deposits'])
n_east = int(event_attrs['n_east'])
n_west = int(event_attrs['n_west'])

print(f'Event {EVENT_INDEX}: {n_deposits:,} deposits (east={n_east:,}, west={n_west:,})\n')

for (s, p) in PLANE_KEYS:
    name = PLANE_NAMES[(s, p)]
    arr = dense_signals[(s, p)]
    n_pix = np.count_nonzero(arr)
    mx = np.abs(arr).max()
    print(f'  {name}: {n_pix:,} pixels, max={mx:.1f} ADC')

In [ ]:
fig = visualize_wire_signals(
    dense_signals, viz_config,
    threshold_enc=THRESHOLD_ENC, gamma=0.2,
)
fig.suptitle(f'{DATASET} file {FILE_INDEX:04d}, event {EVENT_INDEX}', fontsize=14, y=1.02)
plt.show()

## Load Segments (3D Truth Deposits)

In [ ]:
# =============================================================================
# LOAD SEGMENT DATA
# =============================================================================

if has_seg:
    seg = load_event_seg(seg_path, EVENT_INDEX)
    
    n_deps = len(seg['de'])
    n_tracks = len(np.unique(seg['track_ids']))
    n_groups = seg['n_groups']
    total_de = float(seg['de'].sum())
    
    print(f'Segments: {n_deps:,} deposits, {n_tracks:,} tracks, {n_groups:,} groups')
    print(f'Total dE: {total_de:.2f} MeV')
    print(f'Position range: x=[{seg["positions_mm"][:,0].min():.0f}, {seg["positions_mm"][:,0].max():.0f}] mm')
    if seg['qs_fractions'] is not None:
        print(f'Q_s fractions: min={seg["qs_fractions"].min():.4f}, max={seg["qs_fractions"].max():.4f}')
else:
    seg = None
    print('Segment file not found.')

## Load Correspondence and Derive Track Labels

In [ ]:
# =============================================================================
# LOAD CORRESPONDENCE -> TRACK LABELS + DIFFUSED CHARGE
# =============================================================================

track_hits = None
truth_dense = None

if has_corr:
    track_hits, truth_dense, group_to_track = load_event_corr(
        corr_path, EVENT_INDEX, num_time_steps)
    
    for (s, p), name in PLANE_NAMES.items():
        nl = int(track_hits[(s, p)]['num_labeled'])
        print(f'  {name}: {nl:,} labeled pixels')
    
    print(f'\nTrack labels and diffused charge derived from correspondence.')
else:
    print('Correspondence file not found — track labels and diffused charge unavailable.')

## Visualize Diffused Charge (Truth Hits)

In [ ]:
if truth_dense is not None:
    fig = visualize_diffused_charge(truth_dense, viz_config, log_norm=True, threshold=50)
    fig.suptitle(f'{DATASET} event {EVENT_INDEX} — Diffused Charge (from correspondence)',
                 fontsize=14, y=1.02)
    plt.show()
else:
    print('No correspondence data for diffused charge visualization.')

## Visualize Track Labels

In [ ]:
if track_hits is not None:
    top_tracks = get_top_tracks_by_charge(track_hits, top_n=20)
    
    if top_tracks:
        print('Top 10 tracks by charge:')
        for i, (tid, charge) in enumerate(top_tracks[:10]):
            print(f'  {i+1:2d}. Track {tid:4d}: {charge:12,.1f}')
    
    fig = visualize_track_labels(track_hits, viz_config, top_tracks, max_tracks=15)
    fig.suptitle(f'{DATASET} event {EVENT_INDEX} — Track Labels ({n_tracks:,} tracks)',
                 fontsize=14, y=1.02)
    plt.show()
else:
    print('No track labels available.')

## Single Track Visualization

In [ ]:
TRACK_RANK = 1  # Nth most active track (1-indexed)

if track_hits is not None and top_tracks and TRACK_RANK <= len(top_tracks):
    selected_tid, selected_charge = top_tracks[TRACK_RANK - 1]
    print(f'Rank {TRACK_RANK}: Track {selected_tid}, charge={selected_charge:,.1f}')
    
    single_dense = {}
    total_track_hits = 0
    
    for (s, p) in PLANE_KEYS:
        nw = viz_config.side_geom[s].num_wires[p]
        dense = np.zeros((nw, num_time_steps), dtype=np.float32)
        
        data = track_hits[(s, p)]
        nl = int(data['num_labeled'])
        if nl > 0:
            labeled = np.asarray(data['labeled_hits'][:nl])
            tids = np.asarray(data['labeled_track_ids'][:nl])
            mask = tids == selected_tid
            
            if mask.any():
                wire = labeled[mask, 0].astype(np.int32)
                time_idx = labeled[mask, 1].astype(np.int32)
                charge = labeled[mask, 2]
                valid = (wire >= 0) & (wire < nw) & (time_idx >= 0) & (time_idx < num_time_steps)
                np.add.at(dense, (wire[valid], time_idx[valid]), charge[valid])
                total_track_hits += int(mask.sum())
        
        single_dense[(s, p)] = dense
    
    print(f'Track {selected_tid}: {total_track_hits:,} labeled hits')
    
    fig = visualize_diffused_charge(single_dense, viz_config, log_norm=True, threshold=50)
    fig.suptitle(
        f'{DATASET} event {EVENT_INDEX} — Rank #{TRACK_RANK} Track {selected_tid} ({total_track_hits:,} hits)',
        fontsize=14, y=1.02)
    plt.show()
else:
    print('Track labels not available or rank not available.')

## Summary

In [ ]:
print('=' * 60)
print(f' Event Summary: {DATASET} file {FILE_INDEX:04d}, event {EVENT_INDEX}')
print('=' * 60)
print(f'  Deposits:    {n_deposits:,}')
if has_seg:
    print(f'  Tracks:      {n_tracks:,}')
    print(f'  Groups:      {n_groups:,}')
    print(f'  Total dE:    {total_de:.2f} MeV')

total_pix = sum(np.count_nonzero(dense_signals[(s,p)]) for s,p in PLANE_KEYS)
print(f'  Resp pixels: {total_pix:,} (file threshold={threshold_adc_file:.1f} ADC)')

if track_hits is not None:
    total_labeled = sum(int(d['num_labeled']) for d in track_hits.values())
    print(f'  Labeled:     {total_labeled:,} pixels')

print(f'\n  Physics: v={velocity} cm/us, tau={lifetime} us, recomb={recomb_model}')
print(f'  Pipeline: noise={include_noise}, electronics={include_electronics}, digitize={include_digitize}')